In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes rouge-score sentencepiece

import os
os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN"
from huggingface_hub import login, whoami
login(token=os.environ["HF_TOKEN"])
info = whoami()
print(f"✅ Logged in as: {info['name']}")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


HfHubHTTPError: Invalid user token. The token from HF_TOKEN environment variable is invalid. Note that HF_TOKEN takes precedence over `hf auth login`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import whoami
import torch

info = whoami()
username = info['name']

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
your_model = f"{username}/tinyllama-medical-finetuned"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

ft_model = PeftModel.from_pretrained(base_model, your_model)
print("✅ Both models ready!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/3.08M [00:00<?, ?B/s]

✅ Both models ready!


In [ ]:
from rouge_score import rouge_scorer
from datasets import load_dataset
import json, os, torch

scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)

dataset = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
test_examples = dataset['train'].select(range(2000, 2100))  # 100 test examples

def get_answer(the_model, question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = the_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    input_len = inputs["input_ids"].shape[1]
    answer = tokenizer.decode(out[0][input_len:], skip_special_tokens=True)
    return answer.strip()

base_scores, ft_scores = [], []

print("Evaluating 100 questions — takes ~15 mins...")
for i, ex in enumerate(test_examples):
    q   = ex['input']
    ref = ex['output']

    base_ans = get_answer(base_model, q)
    ft_ans   = get_answer(ft_model,   q)

    base_scores.append(scorer.score(ref, base_ans)['rougeL'].fmeasure)
    ft_scores.append(scorer.score(ref, ft_ans)['rougeL'].fmeasure)

    if (i+1) % 20 == 0:
        print(f"  {i+1}/100 done...")

avg_base = round(sum(base_scores)/len(base_scores), 3)
avg_ft   = round(sum(ft_scores)/len(ft_scores),     3)
improvement = round((avg_ft - avg_base)/avg_base*100, 1)

print(f"\n========== YOUR RESULTS ==========")
print(f"Base TinyLlama ROUGE-L:    {avg_base}")
print(f"Your fine-tuned ROUGE-L:   {avg_ft}")
print(f"Improvement:               +{improvement}%")
print(f"==================================")
print(f"\n🎉 This +{improvement}% goes on your resume!")

os.makedirs("results", exist_ok=True)
with open("results/metrics.json","w") as f:
    json.dump({"base":avg_base,"finetuned":avg_ft,"improvement":improvement}, f, indent=2)
print("Saved to results/metrics.json — download this file!")

README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Evaluating 100 questions — takes ~15 mins...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

...

  60/100 done...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

  80/100 done...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

  100/100 done...

========== YOUR RESULTS ==========
Base TinyLlama ROUGE-L:    0.159
Your fine-tuned ROUGE-L:   0.161
Improvement:               +1.3%

🎉 This +1.3% goes on your resume!
Saved to results/metrics.json — download this file!
